# 📝 การบ้าน Day 1: Data Engineering Pipeline
## Agentic RAG: From Zero to Hero

---

### 📋 คำชี้แจง

1. **ให้ทำด้วยตนเอง** — ห้ามใช้ AI ช่วยเขียนโค้ด
2. **ห้ามลอกกัน** — ข้อมูลของแต่ละคนจะ **ไม่เหมือนกัน** (สร้างจากรหัสนักศึกษา)
3. **ส่ง notebook นี้** พร้อมผลลัพธ์ที่ run แล้ว (.ipynb)
4. **คะแนน**: 10 คะแนน

> ⚠️ **ระบบจะตรวจจับการลอก** จากค่า hash, embedding, และ score ที่ต้องตรงกับรหัสนักศึกษาของแต่ละคน

## 📦 ติดตั้ง Dependencies

In [ ]:
%%time
!pip install -q sentence-transformers qdrant-client langchain-text-splitters rank_bm25 pythainlp scikit-learn

import hashlib, os, json, random, numpy as np, re
from sklearn.metrics.pairwise import cosine_similarity
print('✅ ติดตั้งเรียบร้อย!')

## 🎓 กรอกข้อมูลนักศึกษา

In [ ]:
# ─── กรอกข้อมูลของคุณ ───
STUDENT_NAME = ''   # เช่น 'สมชาย ใจดี'
STUDENT_ID   = ''   # เช่น '6512345678'
PHONE        = ''   # เช่น '081-234-5678'
LINE_ID      = ''   # เช่น 'somchai.j'

# ─── ตรวจสอบ (ห้ามแก้ไข) ───
assert len(STUDENT_ID) >= 5, '❌ กรุณากรอกรหัสนักศึกษา!'
assert len(STUDENT_NAME) >= 2, '❌ กรุณากรอกชื่อ-นามสกุล!'

print(f'✅ ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'✅ รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')

## 📄 Step 2: สร้างชุดข้อมูลเฉพาะตัว (ห้ามแก้ไข cell นี้)

In [ ]:
%%time
# ===== ห้ามแก้ไข cell นี้ =====
# สร้างชุดข้อมูลเฉพาะจากรหัสนักศึกษา

random.seed(int(hashlib.md5(STUDENT_ID.encode()).hexdigest()[:8], 16))

# เนื้อหาหลัก — สลับลำดับและเลือกเนื้อหาตามรหัสนักศึกษา
all_paragraphs = [
    'การเรียนรู้ของเครื่อง หรือ Machine Learning เป็นสาขาย่อยของปัญญาประดิษฐ์ที่มุ่งเน้นการพัฒนาอัลกอริทึมที่สามารถเรียนรู้จากข้อมูลและปรับปรุงประสิทธิภาพได้โดยอัตโนมัติ โดยไม่จำเป็นต้องเขียนโปรแกรมอย่างชัดเจนสำหรับทุกกรณี',
    'Deep Learning เป็นเทคนิคหนึ่งของ Machine Learning ที่ใช้โครงข่ายประสาทเทียมหลายชั้น Neural Network ในการประมวลผลข้อมูลที่ซับซ้อน เช่น การจดจำภาพ การแปลภาษา และการสร้างข้อความ',
    'Natural Language Processing หรือ NLP คือสาขาที่เกี่ยวข้องกับการทำให้คอมพิวเตอร์สามารถเข้าใจ ตีความ และสร้างภาษามนุษย์ได้ รวมถึงการวิเคราะห์อารมณ์ การสรุปข้อความ และการตอบคำถาม',
    'Retrieval Augmented Generation หรือ RAG เป็นเทคนิคที่รวมการค้นหาข้อมูลเข้ากับการสร้างข้อความของ LLM เพื่อให้ได้คำตอบที่ถูกต้อง ทันสมัย และอ้างอิงแหล่งข้อมูลได้',
    'Vector Database เป็นฐานข้อมูลที่ออกแบบมาเพื่อจัดเก็บและค้นหาข้อมูลในรูปแบบ Embedding Vector ช่วยให้สามารถค้นหาข้อมูลที่มีความหมายคล้ายคลึงกันได้อย่างรวดเร็ว',
    'Text Embedding คือกระบวนการแปลงข้อความให้เป็นชุดตัวเลขหรือ Vector ที่สามารถแสดงความหมายเชิงความหมายของข้อความนั้นได้ ทำให้คอมพิวเตอร์สามารถเปรียบเทียบความคล้ายคลึงระหว่างข้อความได้',
    'Transformer เป็นสถาปัตยกรรมของ Neural Network ที่ใช้กลไก Attention ในการประมวลผลข้อมูลแบบลำดับ เป็นพื้นฐานของโมเดลภาษาขนาดใหญ่อย่าง GPT BERT และ Gemini',
    'Prompt Engineering คือศาสตร์ของการออกแบบคำสั่งหรือ Prompt ที่ให้กับ LLM เพื่อให้ได้ผลลัพธ์ที่ต้องการ การเขียน Prompt ที่ดีช่วยเพิ่มคุณภาพและความแม่นยำของคำตอบอย่างมาก',
    'Fine-tuning คือกระบวนการปรับแต่งโมเดลที่ผ่านการฝึกมาแล้วด้วยข้อมูลเฉพาะทางเพิ่มเติม เพื่อให้โมเดลทำงานได้ดีขึ้นในงานที่เฉพาะเจาะจง เช่น การวิเคราะห์เอกสารทางการแพทย์',
    'Tokenization คือกระบวนการแบ่งข้อความออกเป็นหน่วยย่อยที่เรียกว่า Token ซึ่งอาจเป็นคำ คำย่อย หรือตัวอักษร สำหรับภาษาไทยการตัดคำมีความซับซ้อนเพราะไม่มีเว้นวรรคระหว่างคำ',
    'Chunking คือกระบวนการแบ่งเอกสารขนาดยาวออกเป็นส่วนย่อยที่เหมาะสมสำหรับการสร้าง Embedding มีหลายวิธีเช่น Fixed-size Recursive และ Semantic Chunking',
    'Cosine Similarity เป็นวิธีวัดความคล้ายคลึงระหว่างสอง Vector โดยดูจากมุมระหว่าง Vector ค่า 1 หมายถึงทิศทางเดียวกัน ค่า 0 หมายถึงตั้งฉาก นิยมใช้ในงาน NLP และ Information Retrieval',
]

# สลับลำดับตาม seed ของนักศึกษา
random.shuffle(all_paragraphs)

# เลือก 8 ย่อหน้า + สร้าง duplicate 1 ตัว
selected = all_paragraphs[:8]
duplicate_idx = random.randint(0, 5)
selected.append(selected[duplicate_idx])  # ย่อหน้าที่ซ้ำ

# บันทึกเป็นไฟล์
os.makedirs('homework_data', exist_ok=True)
for i, para in enumerate(selected):
    with open(f'homework_data/doc_{i+1}.txt', 'w', encoding='utf-8') as f:
        f.write(para)

print(f'✅ สร้างข้อมูลเฉพาะสำหรับ {STUDENT_ID} เรียบร้อย!')
print(f'📁 จำนวนไฟล์: {len(selected)} ไฟล์ (มี 1 ไฟล์ที่ซ้ำ)')
for i in range(len(selected)):
    print(f'  📄 doc_{i+1}.txt ({len(selected[i])} ตัวอักษร)')

---
## 🎯 โจทย์: สร้าง Data Engineering Pipeline

ให้สร้าง **RAG Data Pipeline** จากชุดข้อมูลเฉพาะตัวของคุณ ทำตามขั้นตอนด้านล่าง
แต่ละขั้นตอนมี **คำตอบที่ต้องรายงาน** — ค่าเหล่านี้จะไม่ซ้ำกันระหว่างนักศึกษา

---

### ขั้นตอนที่ 1: ตรวจหา Duplicate ด้วย MD5 (2 คะแนน)

- คำนวณ MD5 hash ของทุกไฟล์ใน `homework_data/`
- ค้นหาไฟล์ที่ซ้ำกัน
- ลบไฟล์ที่ซ้ำออก (เก็บตัวแรก)

**📝 รายงาน:**
1. ไฟล์คู่ไหนที่ซ้ำกัน?
2. MD5 hash ของไฟล์ที่ซ้ำคือเท่าไร?
3. หลังลบแล้วเหลือกี่ไฟล์?

In [ ]:
# ขั้นตอนที่ 1: เขียนโค้ดตรวจหา duplicate ที่นี่

# 💡 Hint:
#   1. import hashlib
#   2. สร้างฟังก์ชัน compute_md5(filepath) ที่อ่านไฟล์แล้ว return hash
#   3. วนลูปทุกไฟล์ใน 'homework_data/' เก็บ hash ใน dict
#   4. ถ้า hash ซ้ำ → แสดงว่าไฟล์ duplicate



# ✅ Self-check (run หลังเขียนโค้ดเสร็จ)
# assert dup_hash is not None, '❌ ยังไม่ได้หาไฟล์ duplicate'
# assert files_remaining >= 7, '❌ จำนวนไฟล์หลังลบน่าจะมากกว่า 7'
# print(f'✅ Step 1 passed: ไฟล์ซ้ำ={dup_files}, MD5={dup_hash}')

### ขั้นตอนที่ 2: Chunking (2 คะแนน)

- รวมข้อความจากไฟล์ที่เหลือ (ไม่รวม duplicate)
- Chunk ด้วย `RecursiveCharacterTextSplitter` — ตั้ง `chunk_size=150`, `chunk_overlap=30`

**📝 รายงาน:**
1. ได้ทั้งหมดกี่ chunks?
2. Chunk ที่ 1 มีเนื้อหาอะไร? (copy มาวาง)
3. Chunk ที่สั้นที่สุดมีกี่ตัวอักษร?

In [ ]:
# ขั้นตอนที่ 2: เขียนโค้ด chunking ที่นี่

# 💡 Hint:
#   1. from langchain_text_splitters import RecursiveCharacterTextSplitter
#   2. รวมเนื้อหาจากไฟล์ที่ไม่ซ้ำ
#   3. สร้าง splitter ด้วย chunk_size=150, chunk_overlap=30
#   4. ใช้ splitter.split_text(all_text)



# ✅ Self-check
# assert len(chunks) > 0, '❌ ยังไม่ได้ chunk'
# assert len(chunks[0]) <= 150, '❌ chunk ใหญ่เกินไป'
# print(f'✅ Step 2 passed: {len(chunks)} chunks, chunk 1 = {len(chunks[0])} chars')

### ขั้นตอนที่ 3: Embedding + Similarity (3 คะแนน)

- สร้าง embedding ด้วย `intfloat/multilingual-e5-large`
- คำนวณ **Cosine Similarity** ระหว่าง query กับทุก chunk
- Query: `'query: เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน'`

**📝 รายงาน:**
1. Chunk ไหนมี similarity สูงสุด? (ระบุเลข chunk และ score ทศนิยม 4 ตำแหน่ง)
2. Chunk ไหนมี similarity ต่ำสุด? (ระบุเลข chunk และ score ทศนิยม 4 ตำแหน่ง)
3. สรุปจากผลลัพธ์ — ทำไม chunk นั้นถึงคล้ายที่สุด? (อธิบายด้วยตัวเอง 2-3 ประโยค)

In [ ]:
# ขั้นตอนที่ 3: เขียนโค้ด embedding + similarity ที่นี่
# อย่าลืมใส่ prefix 'query: ' และ 'passage: ' ตามที่เรียนในคลาส

# 💡 Hint:
#   1. from sentence_transformers import SentenceTransformer
#   2. model = SentenceTransformer('intfloat/multilingual-e5-large')
#   3. query = 'query: เทคนิคการค้นหาข้อมูลที่มีความหมายคล้ายกัน'
#   4. passages = ['passage: ' + c for c in chunks]
#   5. query_emb = model.encode(query)
#   6. passage_embs = model.encode(passages)
#   7. ใช้ cosine_similarity() จาก sklearn



# ✅ Self-check
# assert best_score > 0.5, '❌ similarity score ต่ำเกินไป ลองตรวจ prefix'
# print(f'✅ Step 3 passed: best chunk={best_idx}, score={best_score:.4f}')

### ขั้นตอนที่ 4: Qdrant Retrieval (3 คะแนน)

- สร้าง Qdrant client (in-memory) + Collection ชื่อ `f'hw_{STUDENT_ID}'`
- Upsert ทุก chunk + embedding เข้า Qdrant
- ค้นหาด้วย query: `'query: การแบ่งข้อความออกเป็นส่วนย่อย'`
- ใช้ `top_k=3`

**📝 รายงาน:**
1. ผลลัพธ์ Top-3 แต่ละอันดับ มี score เท่าไร? (ทศนิยม 4 ตำแหน่ง)
2. อันดับ 1 เป็นเนื้อหาเกี่ยวกับอะไร?
3. เขียนสรุป: ถ้าจะนำระบบนี้ไปใช้งานจริง ต้องปรับปรุงอะไรบ้าง? (อธิบาย 3-5 ประโยค)

In [ ]:
# ขั้นตอนที่ 4: เขียนโค้ด Qdrant retrieval ที่นี่

# 💡 Hint:
#   1. from qdrant_client import QdrantClient, models
#   2. qdrant = QdrantClient(':memory:')
#   3. สร้าง collection ชื่อ f'hw_{STUDENT_ID}'
#   4. Upsert ทุก chunk + embedding เข้า Qdrant
#   5. query = 'query: การแบ่งข้อความออกเป็นส่วนย่อย'
#   6. qdrant.query_points(..., limit=3)



# ✅ Self-check
# assert len(results) == 3, '❌ ควรได้ top_k=3 ผลลัพธ์'
# print(f'✅ Step 4 passed: Top-3 scores={[r.score for r in results]}')

## 📊 เกณฑ์การให้คะแนน

| ขั้นตอน | คะแนน | เกณฑ์ |
|---------|:-----:|------|
| 1. หา Duplicates | 2 | ฟังก์ชัน find_duplicates ทำงานได้, ผลถูกต้อง |
| 2. Chunking | 3 | chunk_size/overlap ถูกต้อง, เปรียบเทียบวิธี |
| 3. Embedding + Search | 3 | embed ทำงาน, search ค้นหาได้, ผลสมเหตุสมผล |
| 4. วิเคราะห์ผล | 2 | อธิบายคุณภาพผลลัพธ์, เปรียบเทียบ parameter |
| **รวม** | **10** | |

---
## ✅ ตรวจสอบคำตอบ

Run cell ด้านล่างเพื่อสร้าง **Verification Code** สำหรับส่งงาน

In [ ]:
# ===== ห้ามแก้ไข cell นี้ =====
verify_hash = hashlib.sha256(f'{STUDENT_ID}_day1_hw'.encode()).hexdigest()[:12]
print('=' * 50)
print(f'👤 ชื่อ-นามสกุล: {STUDENT_NAME}')
print(f'🎓 รหัสนักศึกษา: {STUDENT_ID}')
print(f'📱 เบอร์โทร: {PHONE}')
print(f'💬 LINE ID: {LINE_ID}')
print(f'🔑 Verification Code: {verify_hash}')
print(f'📅 ส่งก่อน: _________________ (อาจารย์กำหนด)')
print('=' * 50)
print()
print('📋 Checklist ก่อนส่ง:')
print('  [ ] กรอกข้อมูลส่วนตัวครบถ้วน')
print('  [ ] ขั้นตอนที่ 1: ระบุไฟล์ duplicate + MD5 hash')
print('  [ ] ขั้นตอนที่ 2: ระบุจำนวน chunks + เนื้อหา chunk ที่ 1')
print('  [ ] ขั้นตอนที่ 3: ระบุ chunk ที่คล้ายสุด/ต่างสุด + อธิบาย')
print('  [ ] ขั้นตอนที่ 4: ระบุ Top-3 scores + สรุปการปรับปรุง')
print('  [ ] ทุก cell run แล้วมีผลลัพธ์')